<a href="https://colab.research.google.com/github/antariksha-agi/chatbot-Api/blob/main/chatbot-api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sqlite3
from datetime import datetime
from typing import Optional
from fastapi import FastAPI
from pydantic import BaseModel

# 1. Database Setup
def init_db():
    conn = sqlite3.connect('chatbot.db')
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS interactions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT NOT NULL,
            user_message TEXT NOT NULL,
            bot_reply TEXT NOT NULL,
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    conn.commit()
    conn.close()

init_db()

# 2. Pydantic Models
class messages(BaseModel):
    username: str
    user_message: str

app = FastAPI()

@app.post("/send_message/")
def send_message(msg: messages):
  if msg.username and msg.user_message:
    bot_reply = f"Hello, {msg.username}. You said: {msg.user_message}"
    conn = sqlite3.connect('chatbot.db')
    cursor = conn.cursor()
    cursor.execute('''
    INSERT INTO interactions(username, user_message, bot_reply, timestamp)VALUES(?, ?, ?, ?)''', (msg.username, msg.user_message, bot_reply, datetime.now().isoformat()))
    conn.commit()
    conn.close()
    return {"username": msg.username, "user_message": msg.user_message, "bot_reply": bot_reply, "timestamp": datetime.now()}
  else:
    return "Invalid input"

# Create an instance of the messages Pydantic model
message_instance = messages(username="antariksha", user_message="Hello")
mi2 = messages(username="alex", user_message="yoo wussap")
mi3 = messages(username="anjali", user_message="how are you")
input_data = send_message(msg=message_instance)
input_data2 = send_message(msg=mi2)
input_data3 = send_message(msg=mi3)
print(input_data)
print(input_data2)
print(input_data3)

{'username': 'antariksha', 'user_message': 'Hello', 'bot_reply': 'Hello, antariksha. You said: Hello', 'timestamp': datetime.datetime(2026, 6, 1, 8, 16, 36, 308963)}
{'username': 'alex', 'user_message': 'yoo wussap', 'bot_reply': 'Hello, alex. You said: yoo wussap', 'timestamp': datetime.datetime(2026, 6, 1, 8, 16, 36, 316229)}
{'username': 'anjali', 'user_message': 'how are you', 'bot_reply': 'Hello, anjali. You said: how are you', 'timestamp': datetime.datetime(2026, 6, 1, 8, 16, 36, 323824)}


In [ ]:
@app.delete("/delete_history/{username}")
def delete_history(username: str):
  conn = sqlite3.connect('chatbot.db')
  cursor = conn.cursor()

  cursor.execute("DELETE FROM interactions WHERE username = ?" , (username,))
  conn.commit()
  conn.close()
  return {"message": "History deleted successfully"}
dh = delete_history("alex")
print(dh)

{'message': 'History deleted successfully'}


In [ ]:
@app.get("/history/")
def get_history():
  conn = sqlite3.connect('chatbot.db')
  cursor = conn.cursor()
  cursor.execute('SELECT * FROM interactions')
  rows = cursor.fetchall()
  conn.close()
  return rows

history = get_history()
print(history)

[(17, 'antariksha', 'Hello', 'Hello, antariksha. You said: Hello', '2026-06-01T08:16:36.301064'), (19, 'anjali', 'how are you', 'Hello, anjali. You said: how are you', '2026-06-01T08:16:36.316492')]
